# 实验7.3 Qwen1.5-0.5B大语言模型在昇腾香橙派部署实验

> **昇腾香橙派（Ascend 310B）嵌入式平台 · Qwen1.5-0.5B-Chat · ONNX→OM模型转换 · ACL推理部署 · Gradio Web交互**

本实验在**昇腾香橙派**嵌入式开发板上部署 Qwen1.5-0.5B-Chat 大语言模型。实验流程为：首先在昇腾云沙箱平台上将模型导出为 ONNX 格式（产物已保存在 `code/onnx_export/` 目录下），然后将 ONNX 模型传输到香橙派开发板，使用 **ATC（Ascend Tensor Compiler）** 命令将 ONNX 模型编译为昇腾专用的 **.om 离线模型**，最后通过 `qwen1.5-0.5b-chat.py` 程序调用 **ACL（Ascend Computing Language）** 加载 .om 模型进行推理，并通过 **Gradio** 提供 Web 聊天交互界面。

**运行环境**：昇腾香橙派开发板（Ascend 310B）· Ubuntu 22.04 · CANN 9.0 · Python 3.8+

> **重要说明**：本实验的所有操作均在**昇腾香橙派开发板**上执行，而非在本 Notebook 所在的 PC 上运行。本 Notebook 仅作为**实验指导文档**，将主要命令和代码罗列出来，便于学生学习和理解整个部署流程。学生需按照文档中的步骤，在香橙派开发板的终端中逐一执行相应命令。

---

## 1. 实验概述

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;"><strong>实验名称</strong></td>
<td style="text-align: left;">Qwen1.5-0.5B大语言模型在昇腾香橙派部署实验</td>
</tr>
<tr>
<td style="text-align: left;"><strong>目标硬件</strong></td>
<td style="text-align: left;">昇腾香橙派开发板（Ascend 310B 嵌入式AI芯片）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>操作系统</strong></td>
<td style="text-align: left;">Ubuntu 22.04（ARM架构）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>软件环境</strong></td>
<td style="text-align: left;">CANN 9.0 完整工具包 · Python 3.8+ · pyACL · Transformers · Gradio</td>
</tr>
<tr>
<td style="text-align: left;"><strong>基础模型</strong></td>
<td style="text-align: left;">Qwen1.5-0.5B-Chat（462M 参数）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>模型格式转换</strong></td>
<td style="text-align: left;">ONNX → OM（通过 ATC 命令）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>推理方式</strong></td>
<td style="text-align: left;">ACL 加载 .om 离线模型 + 自回归生成</td>
</tr>
<tr>
<td style="text-align: left;"><strong>交互界面</strong></td>
<td style="text-align: left;">Gradio Web 应用（http://127.0.0.1:7860）</td>
</tr>
</table>

**表格解读**：本实验的核心任务是将云端训练并导出的 Qwen1.5-0.5B-Chat ONNX 模型，部署到资源受限的昇腾香橙派嵌入式平台上。香橙派搭载 Ascend 310B 芯片（面向边缘推理），无法直接运行 PyTorch 模型，需通过华为 ATC 编译器将 ONNX 模型转换为昇腾专用的 .om 离线模型格式。转换后的 .om 模型通过 pyACL 接口加载并在 NPU 上执行推理，最后通过 Gradio 框架提供浏览器聊天界面，实现完整的边缘端大模型对话服务。

### 实验目标

- **知识目标**：理解 ONNX 模型到昇腾 .om 离线模型的转换原理（ATC 编译器）；理解 ACL 推理框架的工作机制（模型加载、数据集创建、推理执行、资源释放）；理解嵌入式端大模型部署的完整流程与约束。
- **能力目标**：能够使用 ATC 命令完成模型格式转换；能够编写基于 pyACL 的推理脚本加载 .om 模型；能够在昇腾香橙派上部署并运行大模型对话服务。
- **素养目标**：建立"云端训练→模型导出→边缘部署→交互服务"的端到端工程思维；理解边缘计算与云端计算的差异与各自优势。

## 2. 部署原理与架构

### 2.1 整体部署架构

```
┌─────────────────────────────────────────────────────────────────┐
│                    昇腾云沙箱平台 (Ascend 910B3)                   │
│                                                                   │
│  Qwen1.5-0.5B-Chat (PyTorch模型)                                  │
│         │                                                          │
│         ▼  torch.onnx.export()                                    │
│  qwen_merged.onnx  (ONNX格式模型)                                 │
│         │                                                          │
└─────────┼──────────────────────────────────────────────────────────┘
          │  SCP/文件传输
          ▼
┌─────────────────────────────────────────────────────────────────┐
│                  昇腾香橙派开发板 (Ascend 310B)                    │
│                                                                   │
│  qwen_merged.onnx                                                 │
│         │                                                          │
│         ▼  ATC命令 (Ascend Tensor Compiler)                       │
│  qwen_merged.om  (昇腾离线模型)                                    │
│         │                                                          │
│         ▼  pyACL加载 + 自回归推理                                  │
│  qwen1.5-0.5b-chat.py                                             │
│         │                                                          │
│         ▼  Gradio Web服务                                          │
│  http://127.0.0.1:7860  (浏览器聊天界面)                           │
│                                                                   │
└─────────────────────────────────────────────────────────────────┘
```

### 2.2 ATC 模型转换原理

**ATC（Ascend Tensor Compiler）** 是华为 CANN 工具包提供的模型编译器，负责将开源框架训练好的模型转换为昇腾 NPU 可高效执行的 **.om（Offline Model）离线模型**。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>--framework=5</code></td>
<td style="text-align: left;">指定输入模型框架为 ONNX（5代表ONNX）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--model</code></td>
<td style="text-align: left;">输入 ONNX 模型文件路径</td>
</tr>
<tr>
<td style="text-align: left;"><code>--output</code></td>
<td style="text-align: left;">输出 .om 模型文件路径（不含.om后缀）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--soc_version</code></td>
<td style="text-align: left;">目标芯片型号，香橙派为 <code>Ascend310B4</code>（或 <code>Ascend310B3</code>，取决于具体型号）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--input_shape</code></td>
<td style="text-align: left;">指定模型输入张量的形状（batch_size, seq_len）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--log=error</code></td>
<td style="text-align: left;">仅输出错误级别日志</td>
</tr>
</table>

**表格解读**：ATC 编译器的核心作用是模型格式适配与算子优化。`--framework=5` 告诉 ATC 输入是 ONNX 格式；`--soc_version` 指定目标芯片型号——这是**最关键的参数**，必须与香橙派实际搭载的芯片一致（310B3 或 310B4），否则转换出的模型无法运行；`--input_shape` 固定输入维度，本实验中 `input_ids:1,32` 表示 batch_size=1、序列长度=32，`attention_mask:1,32` 和 `position_ids:1,32` 同理。ATC 会针对目标芯片的达芬奇架构进行算子融合、内存优化等，生成高效的 .om 离线模型。

### 2.3 ACL 推理流程

```
用户输入文本
    ↓
ChatML模板格式化 (apply_chat_template)
    ↓
Tokenizer编码 (文本 → input_ids)
    ↓
左填充至固定长度 seq_len=32
    ↓  构建 input_ids, attention_mask, position_ids
ACL推理 (acl.mdl.execute)         ← .om模型在Ascend 310B上执行
    ↓
取logits最后一个位置 → argmax → next_token
    ↓
拼接next_token → 重复推理 (自回归生成循环)
    ↓
遇到EOS或达到max_new_tokens → 停止
    ↓
Tokenizer解码 (token_ids → 文本) → 输出回答
```

### 2.4 嵌入式部署的关键约束

- **序列长度固定**：Ascend 310B 的 .om 模型在 ATC 转换时已固定输入形状（seq_len=32），推理时必须将输入填充至该长度，无法动态变化。
- **内存受限**：香橙派内存有限（通常2-4GB），需控制模型大小和序列长度。本实验选择 0.5B 参数的小模型和 seq_len=32 以适应边缘硬件。
- **无 PyTorch 推理**：310B 芯片主要面向推理，不直接运行 PyTorch 模型，必须通过 ACL 加载 .om 离线模型。
- **首次推理较慢**：首次推理需加载模型到 NPU 内存并初始化算子内核，后续推理会更快。

## 3. 代码与文件结构

本实验的代码和模型文件组织如下：

```
Lab7_3/
├── lab7.3_nlp_deploy_orangepi.ipynb   # 本实验指导文档（当前文件）
├── 提示词.txt                           # 实验提示词
└── code/                               # 实验代码与模型目录
    ├── convert_atc.sh                  # ATC模型转换脚本（Shell）
    ├── qwen1.5-0.5b-chat.py            # 推理与Web服务脚本（Python）
    ├── 说明.txt                         # 实验说明文档
    └── onnx_export/                    # ONNX模型导出目录
        ├── qwen_merged.onnx            # 合并后的ONNX模型（主模型文件）
        ├── model.model.embed_tokens.weight  # 嵌入层权重
        ├── onnx__MatMul_*              # 各MatMul算子的权重文件
        └── docx_images/                 # 实验截图（ATC转换、安装依赖、Web界面等）
```

### 关键文件说明

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">位置</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>qwen_merged.onnx</code></td>
<td style="text-align: left;"><code>code/onnx_export/</code></td>
<td style="text-align: left;">在昇腾云沙箱上导出的ONNX格式模型，是ATC转换的输入</td>
</tr>
<tr>
<td style="text-align: left;"><code>convert_atc.sh</code></td>
<td style="text-align: left;"><code>code/</code></td>
<td style="text-align: left;">ATC转换脚本，封装了ONNX→OM的完整命令</td>
</tr>
<tr>
<td style="text-align: left;"><code>qwen1.5-0.5b-chat.py</code></td>
<td style="text-align: left;"><code>code/</code></td>
<td style="text-align: left;">推理主程序，包含ACL推理类、Qwen生成器、Gradio界面</td>
</tr>
</table>

> **注意**：`onnx_export/` 目录下的 `onnx__MatMul_*` 文件和 `model.model.embed_tokens.weight` 是 ONNX 模型的外部权重文件，传输时需**完整复制整个目录**，不可遗漏。

## 4. 部署步骤总览

完整部署流程分为以下6个步骤：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">步骤</th>
<th style="text-align: left;">操作</th>
<th style="text-align: left;">执行位置</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">准备ONNX模型</td>
<td style="text-align: left;">昇腾云沙箱</td>
<td style="text-align: left;">已完成，产物在 <code>code/onnx_export/</code></td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">传输文件到香橙派</td>
<td style="text-align: left;">PC</td>
<td style="text-align: left;">通过SCP/U盘将代码和模型传到开发板</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">配置CANN环境</td>
<td style="text-align: left;">香橙派终端</td>
<td style="text-align: left;">设置ATC和ACL所需的环境变量</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">ATC模型转换</td>
<td style="text-align: left;">香橙派终端</td>
<td style="text-align: left;">执行 <code>convert_atc.sh</code>，ONNX→OM</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">安装Python依赖</td>
<td style="text-align: left;">香橙派终端</td>
<td style="text-align: left;">安装 transformers、gradio、numpy 等</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">启动推理服务</td>
<td style="text-align: left;">香橙派终端</td>
<td style="text-align: left;">运行 <code>qwen1.5-0.5b-chat.py</code>，打开Web界面</td>
</tr>
</table>

下面逐步详细说明每个步骤的操作方法。

## 5. 步骤一：准备ONNX模型

**执行位置**：昇腾云沙箱平台（已完成）

在前序实验中，已在昇腾云沙箱（Ascend 910B3）上将 Qwen1.5-0.5B-Chat 模型导出为 ONNX 格式。导出产物保存在 `code/onnx_export/` 目录下，主要文件包括：

- `qwen_merged.onnx`：合并后的 ONNX 模型主文件
- `onnx__MatMul_*`：各层 MatMul 算子的外部权重文件
- `model.model.embed_tokens.weight`：词嵌入层权重

> **注意**：本步骤已在实验准备阶段完成，学生无需重复操作。只需确认 `code/onnx_export/` 目录下文件完整即可。

## 6. 步骤二：传输文件到香橙派

**执行位置**：PC 终端

将 `code/` 目录下的所有文件（包括 `onnx_export/`、`convert_atc.sh`、`qwen1.5-0.5b-chat.py`）传输到香橙派开发板上。

### 方法一：通过 SCP 传输（推荐）

假设香橙派的 IP 地址为 `192.168.1.100`，用户名为 `HwHiAiUser`：

```bash
# 在PC终端执行，将整个code目录传输到香橙派
scp -r code/ HwHiAiUser@192.168.1.100:~/
```

### 方法二：通过 U 盘传输

1. 将 `code/` 目录复制到 U 盘
2. 将 U 盘插入香橙派 USB 接口
3. 在香橙派终端挂载 U 盘并复制文件：

```bash
# 在香橙派终端执行
sudo mount /dev/sda1 /mnt/usb
cp -r /mnt/usb/code/ ~/
sudo umount /mnt/usb
```

### 传输后验证

在香橙派终端确认文件完整：

In [ ]:
%%bash
# 在香橙派终端执行，验证文件完整性
cd ~/code
ls -la
echo "--- onnx_export目录 ---"
ls -lh onnx_export/qwen_merged.onnx
echo "--- 权重文件数量 ---"
ls onnx_export/ | wc -l

**预期输出**：应能看到 `convert_atc.sh`、`qwen1.5-0.5b-chat.py`、`onnx_export/` 等文件和目录，`qwen_merged.onnx` 文件大小约数百MB，`onnx_export/` 目录下有数十个权重文件。

> **注意**：传输时务必保证 `onnx_export/` 目录完整，缺少任何权重文件都会导致后续 ATC 转换失败。

## 7. 步骤三：配置CANN环境

**执行位置**：香橙派终端

香橙派上需已安装 CANN 9.0 完整工具包（非仅 runtime 版本，ATC 编译器需要完整工具包）。每次打开新终端需先设置 CANN 环境变量。

In [ ]:
%%bash
# 在香橙派终端执行，设置CANN环境变量
# 根据实际安装路径调整，默认路径通常为：
source /usr/local/Ascend/ascend-toolkit/set_env.sh

# 验证ATC命令是否可用
which atc
atc --version

# 验证NPU设备是否正常
npu-smi info

**预期输出**：
- `which atc` 输出 ATC 命令路径（如 `/usr/local/Ascend/ascend-toolkit/latest/bin/atc`）
- `atc --version` 输出 ATC 版本信息
- `npu-smi info` 显示香橙派上 Ascend 310B 芯片的状态信息

> **注意**：若提示 `atc: command not found`，说明 CANN 环境未正确设置或未安装完整工具包。请确认安装了完整版 CANN（含 `atc` 编译器），而非仅 runtime 版本。可将 `source` 命令添加到 `~/.bashrc` 中避免每次手动执行。

## 8. 步骤四：ATC模型转换（ONNX → OM）

**执行位置**：香橙派终端

这是本实验的**核心步骤**——使用 ATC 命令将 ONNX 模型编译为昇腾 .om 离线模型。

### 8.1 ATC转换命令

核心 ATC 命令如下：

```bash
atc --framework=5 \
    --model='./onnx_export/qwen_merged.onnx' \
    --output='./om_export/qwen_merged' \
    --soc_version=Ascend310B4 \
    --input_shape='input_ids:1,32;attention_mask:1,32' \
    --log=error
```

**参数详解**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">值</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>--framework</code></td>
<td style="text-align: left;"><code>5</code></td>
<td style="text-align: left;">5 表示 ONNX 框架</td>
</tr>
<tr>
<td style="text-align: left;"><code>--model</code></td>
<td style="text-align: left;"><code>./onnx_export/qwen_merged.onnx</code></td>
<td style="text-align: left;">输入 ONNX 模型路径</td>
</tr>
<tr>
<td style="text-align: left;"><code>--output</code></td>
<td style="text-align: left;"><code>./om_export/qwen_merged</code></td>
<td style="text-align: left;">输出 OM 模型路径（自动添加 .om 后缀）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--soc_version</code></td>
<td style="text-align: left;"><code>Ascend310B4</code></td>
<td style="text-align: left;"><strong>关键</strong>：香橙派芯片型号，必须与实际硬件匹配</td>
</tr>
<tr>
<td style="text-align: left;"><code>--input_shape</code></td>
<td style="text-align: left;"><code>input_ids:1,32;attention_mask:1,32</code></td>
<td style="text-align: left;">两个输入张量的形状：batch=1, seq_len=32</td>
</tr>
<tr>
<td style="text-align: left;"><code>--log</code></td>
<td style="text-align: left;"><code>error</code></td>
<td style="text-align: left;">仅输出错误日志</td>
</tr>
</table>

> **关键修正说明**：原 `说明.txt` 中的命令使用 `--soc_version=Ascend910B3`（错误！），那是云端训练芯片型号。香橙派使用的是 **Ascend 310B** 芯片，应改为 `--soc_version=Ascend310B4`（本实验香橙派型号为 Ascend 310B4；若为 310B3 则用 `Ascend310B3`）。同时，原命令缺少 `position_ids` 输入，若 ONNX 模型包含该输入节点则需补充。

### 8.2 使用转换脚本 convert_atc.sh

为方便操作，已提供封装好的转换脚本 `code/convert_atc.sh`，其完整内容如下：

In [ ]:
%%bash
#!/bin/bash
# ============================================================
# Qwen1.5-0.5B-Chat ONNX -> OM 模型转换脚本 (昇腾香橙派 310B)
# ============================================================
# 使用说明：
#   1. 确保已安装 CANN 9.0 完整工具包 (非仅runtime)
#   2. 确保已设置环境变量: source ~/.bashrc (含CANN环境)
#   3. 在项目根目录执行: bash convert_atc.sh
# ============================================================

# 设置CANN环境变量 (根据实际安装路径调整)
# source /usr/local/Ascend/ascend-toolkit/set_env.sh

echo "============================================"
echo "  ATC模型转换: ONNX -> OM (Ascend310B)"
echo "============================================"

# 创建输出目录
mkdir -p ./om_export

# 查看ONNX模型信息
echo "[1/2] ONNX模型信息:"
echo "  路径: ./onnx_export/qwen_merged.onnx"
ls -lh ./onnx_export/qwen_merged.onnx

# ============================================================
# 关键修正说明:
#   原命令使用 --soc_version=Ascend910B3 (错误!)
#   香橙派使用的是 Ascend 310B 芯片
#   应改为 --soc_version=Ascend310B4
#
#   若模型导出时包含 position_ids 输入,
#   需在 --input_shape 中补充 position_ids:1,32
# ============================================================

# 执行ATC转换
echo ""
echo "[2/2] 开始ATC转换..."
atc --framework=5 \
    --model='./onnx_export/qwen_merged.onnx' \
    --output='./om_export/qwen_merged' \
    --soc_version=Ascend310B4 \
    --input_shape='input_ids:1,32;attention_mask:1,32' \
    --log=error

# 检查转换结果
if [ -f "./om_export/qwen_merged.om" ]; then
    echo ""
    echo "============================================"
    echo "  转换成功!"
    echo "============================================"
    ls -lh ./om_export/qwen_merged.om
    echo ""
    echo "现在可以运行: python3 qwen1.5-0.5b-chat.py"
else
    echo ""
    echo "============================================"
    echo "  转换失败，请检查错误信息"
    echo "============================================"
    echo "常见问题:"
    echo "  1. soc_version不匹配 -> 确认芯片型号(Ascend310B3或Ascend310B4)"
    echo "  2. input_shape缺少position_ids -> 检查ONNX模型输入节点"
    echo "  3. 内存不足 -> 增大swap或减小seq_len"
    echo "  4. CANN环境未设置 -> source set_env.sh"
fi


### 8.3 执行转换

在香橙派终端的 `code/` 目录下执行转换脚本：

In [ ]:
%%bash
# 在香橙派终端执行
cd ~/code

# 方式一：使用脚本（推荐）
bash convert_atc.sh

# 方式二：直接执行ATC命令
# mkdir -p ./om_export
# atc --framework=5 \
#     --model='./onnx_export/qwen_merged.onnx' \
#     --output='./om_export/qwen_merged' \
#     --soc_version=Ascend310B4 \
#     --input_shape='input_ids:1,32;attention_mask:1,32' \
#     --log=error

**ATC转换结果截图**（在香橙派终端实际运行 `bash convert_atc.sh` 的结果）：

![ATC模型转换结果](code/docx_images/atc_convert_result.png)

从截图可以看到：ONNX 模型 `qwen_merged.onnx` 大小约 977K，ATC 转换成功后生成 `qwen_merged.om` 文件大小约 1.2G。转换过程会输出 `ATC start working now` 和 `ATC run success` 提示信息。

**预期输出**：

```
============================================
  ATC模型转换: ONNX -> OM (Ascend310B)
============================================
[1/2] ONNX模型信息:
  路径: ./onnx_export/qwen_merged.onnx
-rwxr-xr-x 1 ... ... xxxM ... qwen_merged.onnx

[2/2] 开始ATC转换...
ATC start working now, ATC pid is xxxx
ATC run success, success time is xx秒

============================================
  转换成功!
============================================
-rwxr-xr-x 1 ... ... xxxM ... qwen_merged.om
现在可以运行: python3 qwen1.5-0.5b-chat.py
```

转换耗时通常为几分钟到十几分钟（取决于模型大小和开发板性能）。转换成功后，`om_export/qwen_merged.om` 文件即为昇腾离线模型。

> **注意**：
> - 若报 `soc_version` 相关错误，请用 `npu-smi info` 确认芯片型号，将 `Ascend310B3` 改为 `Ascend310B4`（或反之）。
> - 若报 `input_shape` 相关错误，可能 ONNX 模型不包含 `position_ids` 输入，尝试去掉 `;position_ids:1,32`。
> - 若内存不足（OOM），可尝试增大 swap 分区或减小 seq_len。

## 9. 步骤五：安装Python依赖

**执行位置**：香橙派终端

推理脚本 `qwen1.5-0.5b-chat.py` 依赖以下 Python 库：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">依赖库</th>
<th style="text-align: left;">用途</th>
</tr>
<tr>
<td style="text-align: left;"><code>acl</code></td>
<td style="text-align: left;">昇腾计算语言 Python 接口（pyACL），随 CANN 安装</td>
</tr>
<tr>
<td style="text-align: left;"><code>transformers</code></td>
<td style="text-align: left;">HuggingFace 分词器，用于文本编解码</td>
</tr>
<tr>
<td style="text-align: left;"><code>gradio</code></td>
<td style="text-align: left;">Web 交互界面框架</td>
</tr>
<tr>
<td style="text-align: left;"><code>numpy</code></td>
<td style="text-align: left;">数值计算，处理输入输出张量</td>
</tr>
</table>

In [ ]:
%%bash
# 在香橙派终端执行，安装Python依赖

# pyACL 随 CANN 安装，需设置 Python 环境变量
# 若使用 CANN 自带的 Python 环境，acl 已可用

# 安装其他依赖库 (transformers 需指定版本以兼容 Qwen1.5)
pip3 install "transformers==4.39.3" gradio numpy

# 安装 huggingface_hub 用于下载分词器
pip3 install huggingface_hub

# 验证安装
python3 -c "import acl; print('acl OK')"
python3 -c "import transformers; print('transformers OK')"
python3 -c "import gradio; print('gradio OK')"
python3 -c "import numpy; print('numpy OK')"

**安装依赖截图**（在香橙派终端实际运行结果）：

![pip安装transformers和gradio](code/docx_images/pip_install_transformers_gradio.png)

![安装huggingface_hub](code/docx_images/pip_install_huggingface_hub.png)

**预期输出**：四个库均输出 `OK`，表示依赖安装成功。

> **注意**：
> - 若 `import acl` 失败，需确认 CANN 的 Python 环境变量已设置。通常需执行 `source set_env.sh` 并确保 Python 路径包含 CANN 的 `python/site-packages`。
> - 香橙派为 ARM 架构，部分库可能需从源码编译安装，耗时较长。

### 9.1 下载 Qwen 分词器

推理脚本需要 Qwen1.5-0.5B-Chat 的分词器文件（`tokenizer.json`、`vocab.json`、`merges.txt` 等）。
在香橙派上通过 HuggingFace 镜像站下载分词器文件（仅需分词器，无需下载完整模型权重）：

In [ ]:
%%bash
# 在香橙派终端执行，下载 Qwen 分词器文件

# 设置 HuggingFace 镜像站（国内网络环境推荐）
export HF_ENDPOINT=https://hf-mirror.com

# 仅下载分词器相关文件（无需下载完整模型权重）
huggingface-cli download Qwen/Qwen1.5-0.5B-Chat \
    tokenizer_config.json vocab.json merges.txt \
    special_tokens_map.json tokenizer.json added_tokens.json \
    --local-dir ./Qwen1.5-0.5B-Chat

**分词器下载截图**（在香橙派终端实际运行结果）：

![下载Qwen分词器](code/docx_images/hf_download_tokenizer.png)

下载完成后，`./Qwen1.5-0.5B-Chat/` 目录下应包含 `tokenizer.json`、`vocab.json`、`merges.txt` 等分词器文件。

## 10. 步骤六：启动推理服务

**执行位置**：香橙派终端

### 10.1 运行推理脚本

在 `code/` 目录下执行推理主程序：

In [ ]:
%%bash
# 在香橙派终端执行
cd ~/code
python3 qwen1.5-0.5b-chat.py

**预期输出**：

```
============================================================
Qwen1.5-0.5B-Chat 昇腾310B(香橙派) 推理服务
============================================================

[1/3] 正在初始化ACL并加载.om模型...
      模型路径: ./om_export/qwen_merged.om
[ACL] 环境初始化成功, device_id=0
[ACL] 模型加载成功: ./om_export/qwen_merged.om
[ACL] 输入数量=3, 输出数量=1

[2/3] 正在加载Qwen分词器...
      分词器路径: ./Qwen1.5-0.5B-Chat
[Qwen] eos_token_id=151643, pad_token_id=151643
[Qwen] 模型输入数=3, 需要position_ids=True

[3/3] 正在启动Gradio Web服务...
============================================================
服务已启动!
请在浏览器中打开: http://127.0.0.1:7860
按 Ctrl+C 停止服务
============================================================
Running on local URL:  http://0.0.0.0:7860
```

### 10.2 访问Web界面

服务启动后，在浏览器中打开链接：

**http://127.0.0.1:7860**

> 若从 PC 浏览器访问香橙派（非本地），请将 `127.0.0.1` 替换为香橙派的实际 IP 地址，如 `http://192.168.1.100:7860`。

进入网页交互界面 **qwen-interface**，即可开始与 Qwen 模型聊天。

**Gradio Web 界面截图**（在昇腾香橙派实际运行效果）：

![Gradio Web界面](code/docx_images/gradio_web_interface.png)

从截图可以看到 Gradio 聊天界面已成功启动，界面标题为 "Qwen1.5-0.5B-Chat 昇腾310B推理"，下方有消息输入框（Type a message...）、Submit 按钮、Examples 示例问题以及 retry/undo/clear 操作按钮。

## 11. Web界面操作指南

### 11.1 界面布局

打开 http://127.0.0.1:7860 后，将看到 Gradio 聊天界面 **qwen-interface**，主要区域如下：

```
┌──────────────────────────────────────────────────┐
│        Qwen1.5-0.5B-Chat 昇腾310B推理              │
│  基于昇腾香橙派 Ascend 310B 嵌入式平台推理 |        │
│  模型: Qwen1.5-0.5B-Chat | 框架: CANN ACL + OM     │
├──────────────────────────────────────────────────┤
│                                                    │
│              【聊天框 qwen-chat-log】               │
│                                                    │
│        User: 你好，请介绍一下你自己                  │
│        Assistant: 我是Qwen1.5...                    │
│                                                    │
├──────────────────────────────────────────────────┤
│  [Type a message...]                    [Submit]   │  ← 消息输入框
├──────────────────────────────────────────────────┤
│  Examples:                                         │
│  [你好，请介绍一下你自己] [什么是人工智能？]          │
│  [请写一首关于春天的短诗] [1+1等于几？]              │
│  [请用一句话解释什么是深度学习]                      │
├──────────────────────────────────────────────────┤
│  [retry]  [undo]  [clear]                          │  ← 操作按钮
└──────────────────────────────────────────────────┘
```

### 11.2 开始聊天

1. **输入问题**：在页面下方消息输入框 **"Type a message..."** 中输入任何问题，例如 `你好，请介绍一下你自己`。
2. **发送消息**：点击右侧的 **Submit** 按钮（或按回车键）发送消息。
3. **等待回答**：Qwen 模型将对此进行回答。**第一次回答需要较长时间加载**（模型首次推理需初始化 NPU 内核），请耐心等待。
4. **查看回答**：回答将显示在上方聊天框 **qwen-chat-log** 中。

> **提示**：也可直接点击下方 **Examples** 中设置好的问题（如 `什么是人工智能？`、`请写一首关于春天的短诗`、`1+1等于几？` 等），快速发送预设问题。

### 11.3 操作按钮说明

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">按钮</th>
<th style="text-align: left;">功能</th>
<th style="text-align: left;">使用场景</th>
</tr>
<tr>
<td style="text-align: left;"><strong>Submit</strong></td>
<td style="text-align: left;">发送输入框中的消息</td>
<td style="text-align: left;">输入问题后点击发送</td>
</tr>
<tr>
<td style="text-align: left;"><strong>retry</strong></td>
<td style="text-align: left;">重新发送上一条消息，让模型重新回答</td>
<td style="text-align: left;">如果出现 Error，点击 retry 重新生成</td>
</tr>
<tr>
<td style="text-align: left;"><strong>undo</strong></td>
<td style="text-align: left;">撤回上一条消息（用户和助手的回复一起删除）</td>
<td style="text-align: left;">对上一轮对话不满意，撤回重输</td>
</tr>
<tr>
<td style="text-align: left;"><strong>clear</strong></td>
<td style="text-align: left;">清空聊天框中的所有对话</td>
<td style="text-align: left;">开始全新对话</td>
</tr>
</table>

**表格解读**：四个按钮覆盖了聊天交互的完整操作。Submit 发送新消息；retry 在模型回答出错（如显示 Error）时重新生成上一条回复；undo 撤回最近一轮对话（用户提问+助手回复）；clear 清空所有历史对话，重新开始。首次提问时模型加载较慢属正常现象，后续提问会明显加快。

### 11.4 操作流程示例

```
步骤1: 在输入框输入 "什么是人工智能？" → 点击 Submit
        （首次推理加载较慢，等待约10-30秒）
        → 聊天框显示 Qwen 的回答

步骤2: 继续输入 "请写一首关于春天的短诗" → 点击 Submit
        （后续推理较快）
        → 聊天框追加新回答

步骤3: 如果回答出现 Error → 点击 retry
        → 模型重新回答上一条问题

步骤4: 如果想撤回上一轮 → 点击 undo
        → 最后一轮对话被撤回

步骤5: 如果想重新开始 → 点击 clear
        → 聊天框清空，可开始全新对话
```

## 12. 推理脚本源码详解

**文件位置**：`code/qwen1.5-0.5b-chat.py`

推理脚本 `qwen1.5-0.5b-chat.py` 是本实验的核心代码，包含三大模块：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">类/函数</th>
<th style="text-align: left;">功能</th>
</tr>
<tr>
<td style="text-align: left;">ACL推理引擎</td>
<td style="text-align: left;"><code>ACLInference</code> 类</td>
<td style="text-align: left;">ACL环境初始化、.om模型加载、数据集创建、推理执行、资源释放</td>
</tr>
<tr>
<td style="text-align: left;">Qwen对话生成</td>
<td style="text-align: left;"><code>QwenGenerator</code> 类</td>
<td style="text-align: left;">ChatML模板构建、分词、自回归生成、解码</td>
</tr>
<tr>
<td style="text-align: left;">Gradio Web界面</td>
<td style="text-align: left;"><code>create_interface()</code></td>
<td style="text-align: left;">创建聊天界面、设置示例和按钮</td>
</tr>
</table>

### 12.1 配置参数

```python
OM_MODEL_PATH = "./om_export/qwen_merged.om"   # .om离线模型路径
TOKENIZER_PATH = "./Qwen1.5-0.5B-Chat"         # 分词器路径
SEQ_LEN = 32                                   # 输入序列长度（与ATC转换时一致）
MAX_NEW_TOKENS = 28                            # 最大生成token数
DEVICE_ID = 0                                  # NPU设备ID
PORT = 7860                                    # Web服务端口
```

> **关键**：`SEQ_LEN` 必须与 ATC 转换时 `--input_shape` 中的序列长度一致（均为32），否则推理会报形状不匹配错误。

### 12.2 完整源码

以下是 `qwen1.5-0.5b-chat.py` 的完整源码，供学生学习和理解：

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Qwen1.5-0.5B-Chat 模型在昇腾310B(香橙派)上的推理脚本
=====================================================
功能：
  1. 使用ACL(Ascend Computing Language)加载ATC转换后的.om模型
  2. 使用Transformers分词器对输入文本进行编解码
  3. 自回归生成实现对话推理
  4. 通过Gradio提供Web聊天交互界面 (http://127.0.0.1:7860)

运行环境：
  - 硬件：昇腾香橙派 (Ascend 310B)
  - 系统：Ubuntu 22.04
  - 软件：CANN 9.0、Python 3.8+、acl、transformers、gradio、numpy

使用方法：
  python3 qwen1.5-0.5b-chat.py
  然后在浏览器中打开 http://127.0.0.1:7860
"""

import os
import sys
import time
import numpy as np

import acl

from transformers import AutoTokenizer

import gradio as gr


# ==================== 配置参数 ====================

OM_MODEL_PATH = "./om_export/qwen_merged.om"

TOKENIZER_PATH = "./Qwen1.5-0.5B-Chat"

SEQ_LEN = 32

MAX_NEW_TOKENS = 28

DEVICE_ID = 0

PORT = 7860

INPUT_DTYPE = np.int64

OUTPUT_DTYPE = np.float32


# ==================== ACL模型推理类 ====================

class ACLInference:
    """
    基于pyACL的模型推理封装类
    负责ACL环境初始化、模型加载、数据集创建、推理执行和资源释放
    """

    def __init__(self, model_path, device_id=0):
        self.device_id = device_id
        self.model_path = model_path
        self.model_id = None
        self.model_desc = None
        self.context = None
        self._init_acl()
        self._load_model()

    def _init_acl(self):
        """初始化ACL运行环境"""
        ret = acl.init()
        if ret != 0:
            raise RuntimeError("acl.init 失败, ret={}".format(ret))

        ret = acl.rt.set_device(self.device_id)
        if ret != 0:
            raise RuntimeError("acl.rt.set_device 失败, ret={}".format(ret))

        self.context, ret = acl.rt.create_context(self.device_id)
        if ret != 0:
            raise RuntimeError("acl.rt.create_context 失败, ret={}".format(ret))

        print("[ACL] 环境初始化成功, device_id={}".format(self.device_id))

    def _load_model(self):
        """加载.om离线模型并获取模型描述信息"""
        if not os.path.exists(self.model_path):
            raise FileNotFoundError("模型文件不存在: {}".format(self.model_path))

        self.model_id, ret = acl.mdl.load_from_file(self.model_path)
        if ret != 0:
            raise RuntimeError("acl.mdl.load_from_file 失败, ret={}".format(ret))

        self.model_desc = acl.mdl.create_desc()
        ret = acl.mdl.get_desc(self.model_desc, self.model_id)
        if ret != 0:
            raise RuntimeError("acl.mdl.get_desc 失败, ret={}".format(ret))

        self.input_num = acl.mdl.get_num_inputs(self.model_desc)
        self.output_num = acl.mdl.get_num_outputs(self.model_desc)

        self.input_sizes = []
        for i in range(self.input_num):
            size = acl.mdl.get_input_size_by_index(self.model_desc, i)
            self.input_sizes.append(size)

        self.output_sizes = []
        for i in range(self.output_num):
            size = acl.mdl.get_output_size_by_index(self.model_desc, i)
            self.output_sizes.append(size)

        print("[ACL] 模型加载成功: {}".format(self.model_path))
        print("[ACL] 输入数量={}, 输出数量={}".format(self.input_num, self.output_num))
        for i in range(self.input_num):
            print("[ACL]   input[{}] size={} bytes".format(i, self.input_sizes[i]))
        for i in range(self.output_num):
            print("[ACL]   output[{}] size={} bytes".format(i, self.output_sizes[i]))

    def _create_dataset(self, buffers_info):
        """
        创建ACL数据集
        buffers_info: [(data_bytes, size), ...] 或 [(None, size), ...] (仅分配不拷贝)
        返回: (dataset, device_ptrs)
        """
        dataset = acl.mdl.create_dataset()
        device_ptrs = []
        for data_bytes, size in buffers_info:
            device_ptr, ret = acl.rt.malloc(size, 0)
            if ret != 0:
                raise RuntimeError("acl.rt.malloc 失败, ret={}".format(ret))
            device_ptrs.append(device_ptr)

            if data_bytes is not None:
                host_ptr = acl.util.numpy_to_ptr(data_bytes)
                ret = acl.rt.memcpy(device_ptr, size, host_ptr, size, 1)
                if ret != 0:
                    raise RuntimeError("acl.rt.memcpy H2D 失败, ret={}".format(ret))

            data_buf = acl.create_data_buffer(device_ptr, size)
            _, ret = acl.mdl.add_dataset_buffer(dataset, data_buf)
            if ret != 0:
                raise RuntimeError("acl.mdl.add_dataset_buffer 失败, ret={}".format(ret))

        return dataset, device_ptrs

    def _destroy_dataset(self, dataset):
        """销毁ACL数据集并释放设备内存"""
        num = acl.mdl.get_dataset_num_buffers(dataset)
        for i in range(num):
            buf = acl.mdl.get_dataset_buffer(dataset, i)
            device_ptr = acl.get_data_buffer_addr(buf)
            if device_ptr:
                acl.rt.free(device_ptr)
            acl.destroy_data_buffer(buf)
        acl.mdl.destroy_dataset(dataset)

    def infer(self, input_arrays):
        """
        执行模型推理
        input_arrays: [np.ndarray, ...] 输入numpy数组列表
        返回: [np.ndarray, ...] 输出numpy数组列表(原始字节)
        """
        acl.rt.set_context(self.context)

        input_info = []
        for arr in input_arrays:
            input_info.append((arr, arr.nbytes))

        input_dataset, _ = self._create_dataset(input_info)

        output_info = [(None, size) for size in self.output_sizes]
        output_dataset, output_ptrs = self._create_dataset(output_info)

        ret = acl.mdl.execute(self.model_id, input_dataset, output_dataset)
        if ret != 0:
            raise RuntimeError("acl.mdl.execute 失败, ret={}".format(ret))

        outputs = []
        for i in range(self.output_num):
            buf = acl.mdl.get_dataset_buffer(output_dataset, i)
            data_ptr = acl.get_data_buffer_addr(buf)
            data_size = acl.get_data_buffer_size_v2(buf)

            output_bytes = np.zeros(data_size, dtype=np.uint8)
            host_ptr = acl.util.numpy_to_ptr(output_bytes)
            ret = acl.rt.memcpy(host_ptr, data_size, data_ptr, data_size, 2)
            if ret != 0:
                raise RuntimeError("acl.rt.memcpy D2H 失败, ret={}".format(ret))
            outputs.append(output_bytes)

        self._destroy_dataset(input_dataset)
        self._destroy_dataset(output_dataset)

        return outputs

    def get_output_array(self, output_bytes, dtype=np.float32):
        """将原始输出字节转换为numpy数组"""
        return np.frombuffer(output_bytes.tobytes(), dtype=dtype)

    def __del__(self):
        """释放ACL资源"""
        try:
            if self.model_id is not None:
                acl.mdl.unload(self.model_id)
            if self.model_desc is not None:
                acl.mdl.destroy_desc(self.model_desc)
            if self.context is not None:
                acl.rt.destroy_context(self.context)
            acl.rt.reset_device(self.device_id)
            acl.finalize()
            print("[ACL] 资源已释放")
        except Exception:
            pass


# ==================== Qwen对话生成类 ====================

class QwenGenerator:
    """
    Qwen1.5-0.5B-Chat 对话生成器
    负责构建对话prompt、分词、自回归生成、解码
    """

    def __init__(self, model, tokenizer, seq_len, max_new_tokens):
        self.model = model
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.max_new_tokens = max_new_tokens

        self.eos_token_id = tokenizer.eos_token_id
        if self.eos_token_id is None:
            self.eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

        self.needs_position_ids = (model.input_num >= 3)

        self.pad_token_id = tokenizer.pad_token_id
        if self.pad_token_id is None:
            self.pad_token_id = tokenizer.eos_token_id

        print("[Qwen] eos_token_id={}, pad_token_id={}".format(
            self.eos_token_id, self.pad_token_id))
        print("[Qwen] 模型输入数={}, 需要position_ids={}".format(
            model.input_num, self.needs_position_ids))

    def _build_chat_prompt(self, message, history):
        """
        构建ChatML格式的对话prompt
        history: [(user_msg, assistant_msg), ...] 历史对话
        message: 当前用户输入
        """
        messages = []
        for user_msg, assistant_msg in history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": assistant_msg})
        messages.append({"role": "user", "content": message})

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        return prompt

    def _prepare_inputs(self, token_ids):
        """
        将token列表转换为模型输入张量(左填充至seq_len)
        返回: [input_ids, attention_mask] 或 [input_ids, attention_mask, position_ids]
        """
        tokens = token_ids[-self.seq_len:]
        actual_len = len(tokens)
        pad_len = self.seq_len - actual_len

        padded_ids = np.array(
            [self.pad_token_id] * pad_len + tokens,
            dtype=INPUT_DTYPE
        ).reshape(1, self.seq_len)

        attention_mask = np.array(
            [0] * pad_len + [1] * actual_len,
            dtype=INPUT_DTYPE
        ).reshape(1, self.seq_len)

        inputs = [padded_ids, attention_mask]

        if self.needs_position_ids:
            position_ids = np.array(
                [0] * pad_len + list(range(actual_len)),
                dtype=INPUT_DTYPE
            ).reshape(1, self.seq_len)
            inputs.append(position_ids)

        return inputs

    def generate(self, message, history):
        """
        自回归生成对话回复(生成器函数，支持流式输出)
        message: 用户输入文本
        history: 历史对话列表 [(user, assistant), ...]
        yield: 逐步生成的回复文本
        """
        prompt = self._build_chat_prompt(message, history)
        prompt_ids = self.tokenizer.encode(prompt)

        print("[Qwen] prompt token数={}, 将生成最多{}个token".format(
            len(prompt_ids), self.max_new_tokens))

        all_tokens = list(prompt_ids)
        generated_tokens = []
        response_text = ""

        for step in range(self.max_new_tokens):
            inputs = self._prepare_inputs(all_tokens)

            outputs = self.model.infer(inputs)

            logits = self.model.get_output_array(outputs[0], dtype=OUTPUT_DTYPE)

            output_element_count = logits.size
            vocab_size = output_element_count // self.seq_len
            logits = logits.reshape(self.seq_len, vocab_size)

            next_token = int(np.argmax(logits[-1]))

            if next_token == self.eos_token_id:
                print("[Qwen] 生成结束(EOS), 共生成{}个token".format(len(generated_tokens)))
                break

            generated_tokens.append(next_token)
            all_tokens.append(next_token)

            new_text = self.tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            )

            if new_text != response_text:
                response_text = new_text
                yield response_text

        if not response_text:
            response_text = "(模型未生成有效回复，请尝试缩短输入或重试)"
            yield response_text

        print("[Qwen] 最终回复: {}".format(response_text[:100]))


# ==================== Gradio Web界面 ====================

def create_interface(generator):
    """
    创建Gradio聊天界面
    包含: 聊天框、消息输入框、Examples示例、Submit/Retry/Undo/Clear按钮
    """
    examples = [
        "你好，请介绍一下你自己",
        "什么是人工智能？",
        "请写一首关于春天的短诗",
        "1+1等于几？",
        "请用一句话解释什么是深度学习",
    ]

    custom_css = """
    .gradio-container {
        max-width: 800px !important;
    }
    #component-0 {
        border-radius: 12px;
    }
    """

    demo = gr.ChatInterface(
        fn=generator.generate,
        title="Qwen1.5-0.5B-Chat 昇腾310B推理",
        description=(
            "基于昇腾香橙派 Ascend 310B 嵌入式平台推理 | "
            "模型: Qwen1.5-0.5B-Chat | "
            "框架: CANN ACL + OM离线模型"
        ),
        examples=examples,
        theme=gr.themes.Soft(),
        css=custom_css,
        retry_btn="retry",
        undo_btn="undo",
        clear_btn="clear",
        textbox=gr.Textbox(
            placeholder="Type a message...",
            scale=7,
            lines=2,
        ),
        submit_btn="Submit",
    )

    return demo


# ==================== 主函数 ====================

def main():
    print("=" * 60)
    print("Qwen1.5-0.5B-Chat 昇腾310B(香橙派) 推理服务")
    print("=" * 60)

    print("\n[1/3] 正在初始化ACL并加载.om模型...")
    print("      模型路径: {}".format(OM_MODEL_PATH))
    model = ACLInference(OM_MODEL_PATH, device_id=DEVICE_ID)

    print("\n[2/3] 正在加载Qwen分词器...")
    print("      分词器路径: {}".format(TOKENIZER_PATH))
    tokenizer = AutoTokenizer.from_pretrained(
        TOKENIZER_PATH,
        trust_remote_code=True
    )

    generator = QwenGenerator(
        model=model,
        tokenizer=tokenizer,
        seq_len=SEQ_LEN,
        max_new_tokens=MAX_NEW_TOKENS
    )

    print("\n[3/3] 正在启动Gradio Web服务...")
    demo = create_interface(generator)

    print("\n" + "=" * 60)
    print("服务已启动!")
    print("请在浏览器中打开: http://127.0.0.1:{}".format(PORT))
    print("按 Ctrl+C 停止服务")
    print("=" * 60 + "\n")

    demo.launch(
        server_name="0.0.0.0",
        server_port=PORT,
        share=False,
        inbrowser=False,
    )


if __name__ == "__main__":
    main()


### 12.3 代码结构解读

**ACLInference 类**（ACL推理引擎）：
- `_init_acl()`：调用 `acl.init()` 初始化 ACL 运行环境，`acl.rt.set_device()` 设置 NPU 设备，`acl.rt.create_context()` 创建执行上下文。
- `_load_model()`：调用 `acl.mdl.load_from_file()` 从文件加载 .om 离线模型，获取模型描述信息（输入/输出数量和大小）。
- `_create_dataset()` / `_destroy_dataset()`：创建和销毁 ACL 数据集，负责设备内存分配（`acl.rt.malloc`）和数据拷贝（`acl.rt.memcpy`，H2D=主机到设备，D2H=设备到主机）。
- `infer()`：核心推理方法，将输入 numpy 数组转为字节拷贝到设备内存，调用 `acl.mdl.execute()` 执行推理，再将输出拷回主机。

**QwenGenerator 类**（对话生成器）：
- `_build_chat_prompt()`：使用 `apply_chat_template` 构建 ChatML 格式的对话 prompt，支持多轮历史对话。
- `_prepare_inputs()`：将 token 列表左填充至固定长度 `seq_len=32`，生成 `input_ids`、`attention_mask` 和 `position_ids` 三个输入张量。
- `generate()`：自回归生成循环——每步将当前所有 token 送入模型推理，取输出 logits 最后一个位置的 argmax 作为下一个 token，拼接后重复，直到遇到 EOS 或达到最大生成长度。使用 `yield` 实现流式输出。

**create_interface() 函数**（Gradio界面）：
- 使用 `gr.ChatInterface` 创建聊天界面，设置标题、描述、示例问题（Examples）、主题和按钮。
- `retry_btn="retry"`、`undo_btn="undo"`、`clear_btn="clear"` 分别对应重试、撤回、清空按钮。
- `submit_btn="Submit"` 为发送按钮，`placeholder="Type a message..."` 为输入框提示文字。

**main() 函数**（主流程）：
1. 初始化 ACL 并加载 .om 模型
2. 加载 Qwen 分词器
3. 创建对话生成器
4. 启动 Gradio Web 服务（端口 7860）

## 13. 常见问题与故障排除

### 13.1 ATC转换阶段

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">问题</th>
<th style="text-align: left;">原因</th>
<th style="text-align: left;">解决方案</th>
</tr>
<tr>
<td style="text-align: left;"><code>atc: command not found</code></td>
<td style="text-align: left;">CANN环境未设置</td>
<td style="text-align: left;">执行 <code>source /usr/local/Ascend/ascend-toolkit/set_env.sh</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>soc_version</code> 报错</td>
<td style="text-align: left;">芯片型号不匹配</td>
<td style="text-align: left;">用 <code>npu-smi info</code> 确认型号，改为 <code>Ascend310B3</code> 或 <code>Ascend310B4</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>input_shape</code> 报错</td>
<td style="text-align: left;">输入节点不匹配</td>
<td style="text-align: left;">检查ONNX模型输入，尝试去掉或调整 <code>position_ids</code></td>
</tr>
<tr>
<td style="text-align: left;">内存不足(OOM)</td>
<td style="text-align: left;">开发板内存有限</td>
<td style="text-align: left;">增大swap分区，或减小 <code>seq_len</code>（如改为16）</td>
</tr>
<tr>
<td style="text-align: left;">权重文件找不到</td>
<td style="text-align: left;">传输不完整</td>
<td style="text-align: left;">确认 <code>onnx_export/</code> 目录下所有文件已完整传输</td>
</tr>
</table>

### 13.2 推理运行阶段

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">问题</th>
<th style="text-align: left;">原因</th>
<th style="text-align: left;">解决方案</th>
</tr>
<tr>
<td style="text-align: left;"><code>ModuleNotFoundError: No module named 'acl'</code></td>
<td style="text-align: left;">pyACL未正确配置</td>
<td style="text-align: left;">设置CANN Python环境变量，确认 <code>set_env.sh</code> 已执行</td>
</tr>
<tr>
<td style="text-align: left;"><code>FileNotFoundError: 模型文件不存在</code></td>
<td style="text-align: left;">.om模型路径错误</td>
<td style="text-align: left;">确认 <code>./om_export/qwen_merged.om</code> 存在，在 <code>code/</code> 目录下运行</td>
</tr>
<tr>
<td style="text-align: left;"><code>shape mismatch</code> 错误</td>
<td style="text-align: left;">seq_len不一致</td>
<td style="text-align: left;">确认代码中 <code>SEQ_LEN</code> 与ATC转换时的 <code>--input_shape</code> 一致</td>
</tr>
<tr>
<td style="text-align: left;">Web页面无法打开</td>
<td style="text-align: left;">端口被占用或网络问题</td>
<td style="text-align: left;">检查7860端口是否被占用，确认香橙派IP和防火墙设置</td>
</tr>
<tr>
<td style="text-align: left;">首次回答很慢</td>
<td style="text-align: left;">模型首次推理初始化</td>
<td style="text-align: left;">正常现象，耐心等待，后续推理会加快</td>
</tr>
<tr>
<td style="text-align: left;">回答出现 Error</td>
<td style="text-align: left;">推理异常</td>
<td style="text-align: left;">点击 <strong>retry</strong> 按钮重试，或缩短输入文本</td>
</tr>
</table>

### 13.3 端口说明

本实验 Web 服务使用 **7860** 端口（Gradio 默认端口）。若 7860 端口被占用，可修改 `qwen1.5-0.5b-chat.py` 中的 `PORT = 7860` 为其他可用端口（如 6000、8080 等），并相应修改浏览器访问地址。

## 14. 完整操作流程速查

以下是在香橙派上从零开始部署的完整命令序列，供快速参考：

```bash
# ===== 在香橙派终端执行 =====

# 1. 设置CANN环境
source /usr/local/Ascend/ascend-toolkit/set_env.sh

# 2. 进入代码目录
cd ~/code

# 3. ATC模型转换 (ONNX → OM)
bash convert_atc.sh
# 或直接执行ATC命令:
# mkdir -p ./om_export
# atc --framework=5 \
#     --model='./onnx_export/qwen_merged.onnx' \
#     --output='./om_export/qwen_merged' \
#     --soc_version=Ascend310B4 \
#     --input_shape='input_ids:1,32;attention_mask:1,32' \
#     --log=error

# 4. 安装Python依赖
pip3 install "transformers==4.39.3" gradio numpy

# 5. 启动推理服务
python3 qwen1.5-0.5b-chat.py

# 6. 在浏览器中打开
#    http://127.0.0.1:7860 (香橙派本地)
#    或 http://<香橙派IP>:7860 (从PC访问)

# 7. 在聊天界面中输入问题，点击Submit开始对话
#    首次回答较慢，请耐心等待
```

## 15. 实验总结

### 15.1 实验成果

通过本实验，学生完成了 Qwen1.5-0.5B-Chat 大语言模型在昇腾香橙派嵌入式平台上的完整部署，包括：

1. **模型格式转换**：使用 ATC 编译器将 ONNX 模型转换为昇腾 .om 离线模型，理解了 `--soc_version`、`--input_shape` 等关键参数的作用。
2. **ACL推理部署**：通过 pyACL 接口加载 .om 模型并在 Ascend 310B NPU 上执行推理，理解了 ACL 的环境初始化、模型加载、数据集创建、推理执行和资源释放的完整流程。
3. **Web交互服务**：通过 Gradio 框架提供浏览器聊天界面，实现了边缘端大模型对话服务，理解了 Submit、retry、undo、clear 等交互操作。

### 15.2 关键知识点

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">知识点</th>
<th style="text-align: left;">要点</th>
</tr>
<tr>
<td style="text-align: left;">ATC模型转换</td>
<td style="text-align: left;">ONNX→OM，需指定正确的 <code>soc_version</code> 和 <code>input_shape</code></td>
</tr>
<tr>
<td style="text-align: left;">ACL推理框架</td>
<td style="text-align: left;"><code>acl.init</code>→<code>load_from_file</code>→<code>create_dataset</code>→<code>execute</code>→<code>memcpy</code>→资源释放</td>
</tr>
<tr>
<td style="text-align: left;">自回归生成</td>
<td style="text-align: left;">循环推理，每步取 logits 末位 argmax 作为下一 token，直至 EOS</td>
</tr>
<tr>
<td style="text-align: left;">嵌入式约束</td>
<td style="text-align: left;">固定序列长度、内存受限、无PyTorch推理、首次推理较慢</td>
</tr>
<tr>
<td style="text-align: left;">边缘部署优势</td>
<td style="text-align: left;">低功耗、离线运行、数据隐私、实时响应</td>
</tr>
</table>

### 15.3 思考与拓展

- 尝试调整 `seq_len` 和 `max_new_tokens` 参数，观察对生成质量和推理速度的影响。
- 思考：为什么嵌入式端需要将序列长度固定，而云端 PyTorch 推理可以动态变化？
- 拓展：尝试部署其他小模型（如 Qwen1.5-0.5B 的量化版本）到香橙派，比较推理性能。
- 拓展：研究如何通过增大 swap 或模型量化来部署更大的模型到资源受限的嵌入式平台。

## 16. 课后练习

**第1题**（单选题）ATC 命令中 `--framework=5` 表示输入模型的框架是？

- A. TensorFlow
- B. Caffe
- C. ONNX
- D. PyTorch

In [ ]:
q1 = ''
print(f'第1题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）在昇腾香橙派（Ascend 310B）上部署模型时，ATC 命令的 `--soc_version` 参数应设置为？

- A. Ascend910B3
- B. Ascend310B4
- C. Ascend910B4
- D. Ascend310B1

In [ ]:
q2 = ''
print(f'第2题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）ATC 模型转换的核心作用是？

- A. 将 .om 模型转换为 ONNX 模型
- B. 将 ONNX 模型编译为昇腾 NPU 可高效执行的 .om 离线模型
- C. 对大语言模型进行 LoRA 微调训练
- D. 对模型进行量化压缩以减小体积

In [ ]:
q3 = ''
print(f'第3题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）在本实验的 ACL 自回归推理流程中，生成停止的条件是？

- A. 固定生成 100 个 token 后停止
- B. 遇到 EOS token 或达到最大生成 token 数（MAX_NEW_TOKENS）
- C. 用户手动点击 clear 按钮停止
- D. NPU 内存不足时自动停止

In [ ]:
q4 = ''
print(f'第4题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）关于嵌入式端大模型部署的约束，以下说法**错误**的是？

- A. 序列长度在 ATC 转换时已固定，推理时必须将输入填充至该长度
- B. 香橙派内存有限，需控制模型大小和序列长度
- C. Ascend 310B 芯片可以直接运行 PyTorch 模型进行推理，无需转换为 .om 格式
- D. 首次推理较慢，因为需加载模型到 NPU 内存并初始化算子内核

In [ ]:
q5 = ''
print(f'第5题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_05 import grade
grade(globals())